In [1]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from typing import Any, Optional
from core.llm import EasyLLM
from agent.BasicAgent import BasicAgent

from core import enable_logging

enable_logging("INFO")   # 或 "DEBUG"

In [2]:
from context import BaseContextSource,ContextItem
class MockSource(BaseContextSource):
    """测试用的模拟来源"""

    def __init__(self, name: str, items_data: list):
        self._name = name
        self._items_data = items_data

    def fetch(self, query, max_tokens=0, **kwargs):
        return [
            ContextItem(
                content=d["content"],
                source=self._name,
                priority=d.get("priority", 0.5),
                token_count=d.get("token_count", 10),
            )
            for d in self._items_data
        ]

    @property
    def source_name(self):
        return self._name

class MockMemoryItem:
    def __init__(self, content: str, importance: float = 0.5, memory_id: str = ""):
        self.content = content
        self.importance = importance
        self.id = memory_id


class MockWorkingMemory:
    def __init__(self, memories):
        self._memories = memories

    def get_all_memories(self):
        return list(self._memories)


class MockMemoryManage:
    def __init__(self, memories):
        self.memory_types = {
            "working": MockWorkingMemory(memories)
        }


class MockHistoryLLM:
    def __init__(self, response: str):
        self.response = response
        self.last_messages = None

    def invoke(self, messages, **kwargs):
        self.last_messages = messages
        return self.response


In [3]:
from context import ContextBuilder,TokenBudget,TokenCounter
builder = ContextBuilder(
    budget=TokenBudget(max_tokens=260),
    counter=TokenCounter(chars_per_token=1.0),
)

In [ ]:
builder = ContextBuilder()
builder.add_source(MockSource("rag", [{"content": "检索内容", "token_count": 10}]))

history = [
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好，有什么可以帮你？"},
]

messages = builder.build_messages(
    query="什么是RAG?",
    history=history,
    system_prompt="你是测试助手",
)
messages

In [ ]:
builder = ContextBuilder(
    budget=TokenBudget(max_tokens=260),
    counter=TokenCounter(chars_per_token=1.0),
)
history = [
    {"role": "user", "content": "用户提出了第一轮非常长的需求说明，需要系统记住项目背景和目标。"},
    {"role": "assistant", "content": "助手确认了第一轮需求，并详细解释了计划和限制条件。"},
    {"role": "user", "content": "用户继续补充第二轮长约束，包括工具调用和恢复要求。"},
    {"role": "assistant", "content": "助手总结第二轮长约束，并给出后续实现顺序。"},
    {"role": "user", "content": "用户第三轮继续增加新的限制条件，并要求保持时间顺序和工具链完整。"},
    {"role": "assistant", "content": "助手第三轮记录新的限制条件，并说明不能直接粗暴截断历史。"},
    {"role": "user", "content": "用户第四轮要求会话恢复后继续复用摘要缓存。"},
    {"role": "assistant", "content": "助手第四轮确认恢复时需要保留压缩状态。"},
    {"role": "user", "content": "u5"},
    {"role": "assistant", "content": "a5"},
]

messages = builder.build_messages(
    query="q",
    history=history,
    system_prompt="sys",
)

In [ ]:
messages

In [ ]:
builder = ContextBuilder(
    budget=TokenBudget(max_tokens=70),
    counter=TokenCounter(chars_per_token=1.0),
)
builder.add_source(MockSource("rag", [
    {"content": "R" * 30, "token_count": 30, "priority": 0.9},
]))
history = [
    {"role": "user", "content": "U" * 20},
    {"role": "assistant", "content": "A" * 20},
    {"role": "user", "content": "B" * 20},
    {"role": "assistant", "content": "C" * 20},
]

messages = builder.build_messages(
    query="Q" * 10,
    history=history,
    system_prompt="S" * 20,
)

In [ ]:
builder._counter.count_messages(messages)
messages

In [ ]:
builder = ContextBuilder(
    budget=TokenBudget(max_tokens=260),
    counter=TokenCounter(chars_per_token=1.0),
)
from context import LLMHistoryCompactor
from core.Message import UserMessage,AssistantMessage
from datetime import datetime
from core.llm import EasyLLM
history_c=LLMHistoryCompactor(llm=EasyLLM())
builder.set_history_compactor(history_c)
history = [
UserMessage(role='user', content='使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22', time=datetime(2026, 4, 9, 22, 18, 33, 428088), metadata={}),
 {'type': 'message',
  'role': 'assistant',
  'content': [{'annotations': [],
    'text': '我先用工具翻译文本，并并行计算 \\(3^{22}\\)，然后给你结果并判断翻译工具是否正确。',
    'type': 'output_text',
    'logprobs': []}],
  'phase': 'commentary'},
 {'type': 'function_call',
  'call_id': 'call_oL1bVZvhFn3id0ErDFQDNK9f',
  'name': 'translate_tool',
  'arguments': '{"target_lang":"en","text":"你是谁，在哪里"}'},
 {'type': 'function_call',
  'call_id': 'call_E2V6qUHkPoehkdpHqxrPPTFk',
  'name': 'calculator',
  'arguments': '{"expression":"3**22"}'},
 {'type': 'function_call_output',
  'call_id': 'call_oL1bVZvhFn3id0ErDFQDNK9f',
  'output': 'Translated: 你是谁，在哪里'},
 {'type': 'function_call_output',
  'call_id': 'call_E2V6qUHkPoehkdpHqxrPPTFk',
  'output': '31381059609'},
 AssistantMessage(role='assistant', content='翻译结果：\n- 原文：你是谁，在哪里\n- 工具返回：`你是谁，在哪里`\n- 正确英文应为：`Who are you, and where are you?`\n\n计算结果：\n- \\(3^{22} = 31381059609\\)\n\n关于“这个工具正确吗”的判断：\n- 这次看，**不正确或未正常工作**。\n- 因为它声称进行了英译，但返回内容仍然是中文，没有完成翻译。', time=datetime(2026, 4, 9, 22, 18, 51, 335844), metadata={})]



2026-04-10 20:32:45,033 | INFO | EasyLLM 初始化完成: provider=google, model=gemini-3-flash


TypeError: pydantic.main.BaseModel.__init__() got multiple values for keyword argument 'role'

In [5]:
builder.build_messages(
    query="q",
    history=history,
    system_prompt="sys",
)

2026-04-10 19:21:57,738 | INFO | Compact History
2026-04-10 19:22:31,602 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-10 19:22:31,610 | INFO | ✅ google Provider 响应成功


[{'role': 'system', 'content': 'sys'},
 {'role': 'user',
  'content': '项目背景及目标已明确。核心约束：1. 支持工具调用与恢复，且会话恢复需复用摘要缓存；2. 必须保持时间顺序与工具链完整，严禁粗暴截断历史。'},
 {'role': 'assistant',
  'content': '已确认所有约束、限制及实现顺序。将确保压缩状态在恢复时保持一致，保障任务连续性。目前已记录至第四轮恢复要求。'},
 {'role': 'assistant', 'content': '针对 u5 的响应：a5'},
 {'role': 'user', 'content': 'q'}]

In [17]:
builder = ContextBuilder(
    budget=TokenBudget(max_tokens=30),
    counter=TokenCounter(chars_per_token=1.0),
)
history = [
    {
        "id": "msg_1",
        "type": "message",
        "role": "assistant",
        "phase": "commentary",
        "content": [{"type": "output_text", "text": "先检查天气"}],
    },
    {
        "type": "function_call",
        "call_id": "call_1",
        "name": "weather",
        "arguments": "{\"city\":\"Shanghai\"}",
    },
    {
        "type": "function_call_output",
        "call_id": "call_1",
        "output": "sunny",
    },
    {"role": "user", "content": "继续总结"},
    {"role": "assistant", "content": "已完成"},
]

compacted = builder.compact_history(history, max_tokens=50)
compacted

2026-04-10 20:12:54,240 | INFO | Compact History


[{'id': 'msg_1',
  'type': 'message',
  'role': 'assistant',
  'phase': 'commentary',
  'content': [{'type': 'output_text', 'text': '先检查天气'}]},
 {'type': 'function_call',
  'call_id': 'call_1',
  'name': 'weather',
  'arguments': '{"city":"Shanghai"}'},
 {'type': 'function_call_output', 'call_id': 'call_1', 'output': 'sunny'},
 {'role': 'user', 'content': '继续总结'},
 {'role': 'assistant', 'content': '已完成'}]